<a href="https://colab.research.google.com/github/saacosta-bit/Sinanzas-Agente/blob/main/notebooks/sinanzas_agente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
!pip install -q langchain-google-genai langgraph faiss-cpu langchain-community langchain-text-splitters

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import sqlite3
from datetime import datetime
from zoneinfo import ZoneInfo
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.tools import tool

DB_PATH = "sinanzas.db"
CATEGORIAS = ["Alimentación", "Transporte", "Servicios",
              "Entretenimiento", "Salud", "Educación", "Otros"]

def hoy_chile() -> str:
    return datetime.now(ZoneInfo("America/Santiago")).date().isoformat()

def init_db():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS gastos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                monto REAL NOT NULL,
                categoria TEXT NOT NULL,
                descripcion TEXT,
                fecha TEXT NOT NULL
            )""")

init_db()

class GastoInput(BaseModel):
    monto: float = Field(gt=0, description="Monto del gasto en pesos chilenos")
    categoria: str = Field(description=f"Categoría, una de: {', '.join(CATEGORIAS)}")
    descripcion: str = Field(default="", description="Detalle breve del gasto")
    fecha: Optional[str] = Field(default=None,
        description="Fecha en formato YYYY-MM-DD. Si no se indica, se usa hoy")

@tool("registrar_gasto", args_schema=GastoInput)
def registrar_gasto(monto: float, categoria: str,
                    descripcion: str = "", fecha: Optional[str] = None) -> str:
    """Registra un gasto del usuario en la base de datos.
    Úsala cuando el usuario cuente que gastó dinero en algo."""
    if categoria not in CATEGORIAS:
        categoria = "Otros"
    try:
        fecha = datetime.strptime(fecha, "%Y-%m-%d").date().isoformat() if fecha else hoy_chile()
    except ValueError:
        return "Error: la fecha debe tener formato YYYY-MM-DD."
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO gastos (monto, categoria, descripcion, fecha) VALUES (?, ?, ?, ?)",
            (monto, categoria, descripcion, fecha))
    return f"Gasto registrado: ${monto:,.0f} en {categoria} ({fecha})."

In [ ]:
print(registrar_gasto.invoke({"monto": 5000, "categoria": "Alimentación",
                              "descripcion": "Almuerzo"}))

with sqlite3.connect(DB_PATH) as conn:
    print(conn.execute("SELECT * FROM gastos").fetchall())

Gasto registrado: $5,000 en Alimentación (2026-10-25).
[(1, 5000.0, 'Alimentación', 'Almuerzo', '2026-10-25')]


In [ ]:
def init_tabla_ingresos():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS ingresos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                monto REAL NOT NULL,
                fuente TEXT,
                fecha TEXT NOT NULL
            )""")

init_tabla_ingresos()

class IngresoInput(BaseModel):
    monto: float = Field(gt=0, description="Monto del ingreso en pesos chilenos")
    fuente: str = Field(default="Sueldo", description="Origen del ingreso (sueldo, honorarios, venta, etc.)")
    fecha: Optional[str] = Field(default=None,
        description="Fecha en formato YYYY-MM-DD. Si no se indica, se usa hoy")

@tool("registrar_ingreso", args_schema=IngresoInput)
def registrar_ingreso(monto: float, fuente: str = "Sueldo",
                      fecha: Optional[str] = None) -> str:
    """Registra un ingreso de dinero del usuario.
    Úsala cuando el usuario cuente cuánto ganó o recibió."""
    try:
        fecha = datetime.strptime(fecha, "%Y-%m-%d").date().isoformat() if fecha else hoy_chile()
    except ValueError:
        return "Error: la fecha debe tener formato YYYY-MM-DD."
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO ingresos (monto, fuente, fecha) VALUES (?, ?, ?)",
            (monto, fuente, fecha))
    return f"Ingreso registrado: ${monto:,.0f} por {fuente} ({fecha})."

In [ ]:
def init_tabla_metas():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS metas (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre TEXT NOT NULL,
                monto_objetivo REAL NOT NULL,
                plazo_meses INTEGER NOT NULL,
                fecha_inicio TEXT NOT NULL
            )""")

init_tabla_metas()

class MetaInput(BaseModel):
    nombre: str = Field(description="Nombre de la meta (ej. 'Notebook nuevo')")
    monto_objetivo: float = Field(gt=0, description="Monto total a ahorrar en pesos chilenos")
    plazo_meses: int = Field(gt=0, description="Plazo en meses para lograr la meta")

@tool("crear_meta_ahorro", args_schema=MetaInput)
def crear_meta_ahorro(nombre: str, monto_objetivo: float, plazo_meses: int) -> str:
    """Crea una meta de ahorro del usuario.
    Úsala cuando el usuario diga que quiere ahorrar una cantidad en un plazo."""
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO metas (nombre, monto_objetivo, plazo_meses, fecha_inicio) VALUES (?, ?, ?, ?)",
            (nombre, monto_objetivo, plazo_meses, hoy_chile()))
    mensual = monto_objetivo / plazo_meses
    return (f"Meta creada: '{nombre}' por ${monto_objetivo:,.0f} en {plazo_meses} meses "
            f"(equivale a ${mensual:,.0f} por mes).")

In [ ]:
print(registrar_ingreso.invoke({"monto": 800000, "fuente": "Sueldo"}))
print(crear_meta_ahorro.invoke({"nombre": "Notebook nuevo",
                                "monto_objetivo": 600000, "plazo_meses": 6}))

with sqlite3.connect(DB_PATH) as conn:
    print(conn.execute("SELECT * FROM ingresos").fetchall())
    print(conn.execute("SELECT * FROM metas").fetchall())

Ingreso registrado: $800,000 por Sueldo (2026-10-25).
Meta creada: 'Notebook nuevo' por $600,000 en 6 meses (equivale a $100,000 por mes).
[(1, 800000.0, 'Sueldo', '2026-10-25')]
[(1, 'Notebook nuevo', 600000.0, 6, '2026-10-25')]


In [ ]:
class ConsultaInput(BaseModel):
    categoria: Optional[str] = Field(default=None,
        description=f"Categoría a filtrar (opcional): {', '.join(CATEGORIAS)}")
    mes: Optional[str] = Field(default=None,
        description="Mes a consultar en formato YYYY-MM (opcional). Si no se indica, trae todos")

@tool("consultar_gastos", args_schema=ConsultaInput)
def consultar_gastos(categoria: Optional[str] = None, mes: Optional[str] = None) -> str:
    """Consulta los gastos registrados, con filtro opcional por categoría y mes.
    Devuelve el detalle y el total. Úsala cuando el usuario pregunte cuánto gastó."""
    query = "SELECT fecha, categoria, descripcion, monto FROM gastos WHERE 1=1"
    params = []
    if categoria:
        query += " AND categoria = ?"
        params.append(categoria)
    if mes:
        query += " AND substr(fecha, 1, 7) = ?"
        params.append(mes)
    query += " ORDER BY fecha DESC"

    with sqlite3.connect(DB_PATH) as conn:
        filas = conn.execute(query, params).fetchall()

    if not filas:
        return "No hay gastos registrados con esos filtros."
    total = sum(f[3] for f in filas)
    detalle = "\n".join(f"- {f[0]} | {f[1]} | {f[2] or 'sin descripción'} | ${f[3]:,.0f}" for f in filas)
    return f"{detalle}\nTotal: ${total:,.0f} ({len(filas)} gastos)"

In [ ]:
!pip install -q faiss-cpu langchain-community langchain-text-splitters

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# CONTENIDO DE EJEMPLO: reemplázalo por guías reales y cita sus fuentes en el informe
GUIAS = [
    "Regla 50/30/20: destina 50% del ingreso a necesidades básicas, 30% a gustos personales y 20% a ahorro o pago de deudas.",
    "Gastos hormiga: son gastos pequeños y frecuentes, como cafés o snacks diarios, que sumados al mes pueden representar un monto importante. Anótalos para hacerlos visibles.",
    "Fondo de emergencia: apunta a reunir entre 3 y 6 meses de gastos básicos en una cuenta separada y de fácil acceso.",
    "Para salir de deudas, prioriza pagar primero la de mayor tasa de interés y mantén al día los pagos mínimos de las demás.",
    "Ahorra primero: transfiere un porcentaje fijo de tu ingreso a una cuenta de ahorro apenas lo recibes, en vez de ahorrar lo que sobre a fin de mes.",
    "Antes de una compra no planificada, espera 24 horas. Si sigues queriéndola, revisa si cabe en tu presupuesto del mes.",
]

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.create_documents(GUIAS)
vector_store = FAISS.from_documents(chunks, embeddings)

class GuiaInput(BaseModel):
    pregunta: str = Field(description="Consulta del usuario sobre finanzas personales")

@tool("buscar_guias_financieras", args_schema=GuiaInput)
def buscar_guias_financieras(pregunta: str) -> str:
    """Busca consejos en las guías de educación financiera.
    Úsala cuando el usuario pregunte cómo ahorrar, presupuestar o manejar deudas."""
    resultados = vector_store.similarity_search(pregunta, k=2)
    return "\n".join(f"- {doc.page_content}" for doc in resultados)

/tmp/ipykernel_1570/1551477744.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
registrar_gasto.invoke({"monto": 2500, "categoria": "Alimentación", "descripcion": "Café"})
registrar_gasto.invoke({"monto": 1200, "categoria": "Transporte", "descripcion": "Metro"})

print(consultar_gastos.invoke({}))
print("---")
print(consultar_gastos.invoke({"categoria": "Alimentación", "mes": "2026-10"}))
print("---")
print(buscar_guias_financieras.invoke({"pregunta": "¿Cómo puedo ahorrar más dinero?"}))

- 2026-10-25 | Alimentación | Almuerzo | $5,000
- 2026-10-25 | Alimentación | Café | $2,500
- 2026-10-25 | Transporte | Metro | $1,200
Total: $8,700 (3 gastos)
---
- 2026-10-25 | Alimentación | Almuerzo | $5,000
- 2026-10-25 | Alimentación | Café | $2,500
Total: $7,500 (2 gastos)
---
- Ahorra primero: transfiere un porcentaje fijo de tu ingreso a una cuenta de ahorro apenas lo recibes, en vez de ahorrar lo que sobre a fin de mes.
- Regla 50/30/20: destina 50% del ingreso a necesidades básicas, 30% a gustos personales y 20% a ahorro o pago de deudas.


In [ ]:
class HormigaInput(BaseModel):
    mes: Optional[str] = Field(default=None,
        description="Mes en formato YYYY-MM. Si se omite, se usa el mes actual")
    umbral_monto: float = Field(default=5000, gt=0,
        description="Monto máximo (CLP) para considerar un gasto como pequeño")
    min_repeticiones: int = Field(default=3, ge=2,
        description="Veces mínimas que debe repetirse el gasto en el mes")

@tool("detectar_gastos_hormiga", args_schema=HormigaInput)
def detectar_gastos_hormiga(mes: Optional[str] = None, umbral_monto: float = 5000,
                            min_repeticiones: int = 3) -> str:
    """Detecta gastos hormiga: gastos pequeños que se repiten muchas veces en el mes.
    Úsala cuando el usuario quiera saber en qué se le va el dinero en cosas pequeñas."""
    mes = mes or hoy_chile()[:7]
    with sqlite3.connect(DB_PATH) as conn:
        total_mes = conn.execute(
            "SELECT COALESCE(SUM(monto), 0) FROM gastos WHERE substr(fecha, 1, 7) = ?",
            (mes,)).fetchone()[0]
        filas = conn.execute("""
            SELECT categoria, LOWER(TRIM(descripcion)), COUNT(*), SUM(monto)
            FROM gastos
            WHERE substr(fecha, 1, 7) = ? AND monto <= ?
            GROUP BY categoria, LOWER(TRIM(descripcion))
            HAVING COUNT(*) >= ?
            ORDER BY SUM(monto) DESC""", (mes, umbral_monto, min_repeticiones)).fetchall()

    if not filas:
        return f"No se detectaron gastos hormiga en {mes}."
    suma = sum(f[3] for f in filas)
    pct = (suma / total_mes * 100) if total_mes else 0
    detalle = "\n".join(
        f"- {f[1] or 'sin descripción'} ({f[0]}): {f[2]} veces, ${f[3]:,.0f}" for f in filas)
    return (f"Gastos hormiga en {mes}:\n{detalle}\n"
            f"Total: ${suma:,.0f}, equivale al {pct:.1f}% de tus gastos del mes.")

In [ ]:
import math

CATEGORIAS_FLEXIBLES = ["Entretenimiento", "Alimentación", "Otros"]

class PlanInput(BaseModel):
    nombre_meta: Optional[str] = Field(default=None,
        description="Nombre de la meta. Si se omite, se usa la más reciente")
    mes: Optional[str] = Field(default=None,
        description="Mes a analizar en formato YYYY-MM. Si se omite, el mes actual")

@tool("calcular_plan_ahorro", args_schema=PlanInput)
def calcular_plan_ahorro(nombre_meta: Optional[str] = None, mes: Optional[str] = None) -> str:
    """Calcula si el usuario puede cumplir una meta de ahorro con sus ingresos y gastos
    del mes, y sugiere recortes por categoría si no alcanza."""
    mes = mes or hoy_chile()[:7]
    with sqlite3.connect(DB_PATH) as conn:
        if nombre_meta:
            meta = conn.execute(
                "SELECT nombre, monto_objetivo, plazo_meses FROM metas "
                "WHERE LOWER(nombre) = LOWER(?) ORDER BY id DESC LIMIT 1", (nombre_meta,)).fetchone()
        else:
            meta = conn.execute(
                "SELECT nombre, monto_objetivo, plazo_meses FROM metas ORDER BY id DESC LIMIT 1").fetchone()
        ingresos = conn.execute(
            "SELECT COALESCE(SUM(monto), 0) FROM ingresos WHERE substr(fecha, 1, 7) = ?",
            (mes,)).fetchone()[0]
        por_cat = conn.execute(
            "SELECT categoria, SUM(monto) FROM gastos WHERE substr(fecha, 1, 7) = ? "
            "GROUP BY categoria ORDER BY SUM(monto) DESC", (mes,)).fetchall()

    if not meta:
        return "No hay metas de ahorro registradas. Crea una primero."
    if ingresos == 0:
        return f"No hay ingresos registrados en {mes}. Registra un ingreso para calcular el plan."

    nombre, objetivo, plazo = meta
    gastos = sum(m for _, m in por_cat)
    disponible = ingresos - gastos
    requerido = objetivo / plazo

    resumen = (f"Meta '{nombre}': ${objetivo:,.0f} en {plazo} meses (${requerido:,.0f} por mes). "
               f"Ingresos de {mes}: ${ingresos:,.0f}, gastos: ${gastos:,.0f}, "
               f"disponible: ${disponible:,.0f}.")
    if disponible >= requerido:
        return resumen + " La meta es alcanzable con tu ritmo actual."

    deficit = requerido - disponible
    flexibles = [(c, m) for c, m in por_cat if c in CATEGORIAS_FLEXIBLES]
    total_flex = sum(m for _, m in flexibles)
    sugerencias = []
    for c, m in flexibles:
        recorte = min(deficit * m / total_flex, m) if total_flex else 0
        sugerencias.append(f"- Reducir {c} en ${recorte:,.0f} (gastas ${m:,.0f})")
    texto = f"{resumen}\nTe faltan ${deficit:,.0f} al mes."
    if sugerencias:
        texto += "\nRecortes sugeridos:\n" + "\n".join(sugerencias)
    if total_flex < deficit:
        texto += "\nAun recortando todo lo flexible no alcanza."
    if disponible > 0:
        texto += f"\nAlternativa: ampliar el plazo a {math.ceil(objetivo / disponible)} meses."
    return texto

In [ ]:
for _ in range(5):
    registrar_gasto.invoke({"monto": 2500, "categoria": "Alimentación", "descripcion": "Café"})
registrar_gasto.invoke({"monto": 300000, "categoria": "Servicios", "descripcion": "Arriendo"})
registrar_gasto.invoke({"monto": 250000, "categoria": "Alimentación", "descripcion": "Supermercado"})
registrar_gasto.invoke({"monto": 180000, "categoria": "Entretenimiento", "descripcion": "Salidas"})

print(detectar_gastos_hormiga.invoke({}))
print("---")
print(calcular_plan_ahorro.invoke({}))

Gastos hormiga en 2026-10:
- café (Alimentación): 6 veces, $15,000
Total: $15,000, equivale al 2.0% de tus gastos del mes.
---
Meta 'Notebook nuevo': $600,000 en 6 meses ($100,000 por mes). Ingresos de 2026-10: $800,000, gastos: $751,200, disponible: $48,800.
Te faltan $51,200 al mes.
Recortes sugeridos:
- Reducir Alimentación en $30,720 (gastas $270,000)
- Reducir Entretenimiento en $20,480 (gastas $180,000)
Alternativa: ampliar el plazo a 13 meses.


In [ ]:
def mes_anterior(mes: str) -> str:
    anio, m = map(int, mes.split("-"))
    return f"{anio - 1}-12" if m == 1 else f"{anio}-{m - 1:02d}"

def gasto_por_categoria(mes: str) -> dict:
    with sqlite3.connect(DB_PATH) as conn:
        return dict(conn.execute(
            "SELECT categoria, SUM(monto) FROM gastos "
            "WHERE substr(fecha, 1, 7) = ? GROUP BY categoria", (mes,)).fetchall())

class AlertaInput(BaseModel):
    categoria: Optional[str] = Field(default=None,
        description="Categoría a revisar. Si se omite, se revisan todas")
    mes: Optional[str] = Field(default=None,
        description="Mes en formato YYYY-MM. Si se omite, el mes actual")
    umbral_porcentaje: float = Field(default=30, gt=0,
        description="Aumento porcentual respecto al mes anterior que dispara la alerta")
    limite_monto: Optional[float] = Field(default=None, gt=0,
        description="Límite de gasto en pesos definido por el usuario (opcional)")

@tool("generar_alerta_categoria", args_schema=AlertaInput)
def generar_alerta_categoria(categoria: Optional[str] = None, mes: Optional[str] = None,
                             umbral_porcentaje: float = 30,
                             limite_monto: Optional[float] = None) -> str:
    """Genera alertas cuando el gasto de una categoría supera el límite del usuario
    o aumenta mucho respecto al mes anterior."""
    mes = mes or hoy_chile()[:7]
    prev = mes_anterior(mes)
    actual = gasto_por_categoria(mes)
    anterior = gasto_por_categoria(prev)
    categorias = [categoria] if categoria else list(actual.keys())

    alertas = []
    for c in categorias:
        g, p = actual.get(c, 0), anterior.get(c, 0)
        if p > 0 and g > p * (1 + umbral_porcentaje / 100):
            alertas.append(f"ALERTA en {c}: ${g:,.0f} en {mes}, "
                           f"{(g - p) / p * 100:.0f}% más que en {prev} (${p:,.0f}).")
        if limite_monto and g > limite_monto:
            alertas.append(f"ALERTA en {c}: superaste tu límite de ${limite_monto:,.0f} "
                           f"(llevas ${g:,.0f}).")
    if not alertas:
        return f"Sin alertas en {mes}: tus gastos están dentro de lo esperado."
    return "\n".join(alertas)

In [ ]:
class ReporteInput(BaseModel):
    mes: Optional[str] = Field(default=None,
        description="Mes en formato YYYY-MM. Si se omite, el mes actual")

@tool("generar_reporte_mensual", args_schema=ReporteInput)
def generar_reporte_mensual(mes: Optional[str] = None) -> str:
    """Genera el resumen del mes: ingresos, gastos por categoría, ahorro
    y comparación con el mes anterior."""
    mes = mes or hoy_chile()[:7]
    prev = mes_anterior(mes)
    with sqlite3.connect(DB_PATH) as conn:
        ingresos = conn.execute(
            "SELECT COALESCE(SUM(monto), 0) FROM ingresos WHERE substr(fecha, 1, 7) = ?",
            (mes,)).fetchone()[0]
    por_cat = gasto_por_categoria(mes)
    gastos = sum(por_cat.values())
    gastos_prev = sum(gasto_por_categoria(prev).values())
    ahorro = ingresos - gastos

    lineas = [f"REPORTE {mes}",
              f"Ingresos: ${ingresos:,.0f}",
              f"Gastos: ${gastos:,.0f}",
              f"Ahorro: ${ahorro:,.0f}" +
              (f" ({ahorro / ingresos * 100:.1f}% del ingreso)" if ingresos else ""),
              "Gastos por categoría:"]
    for c, m in sorted(por_cat.items(), key=lambda x: -x[1]):
        lineas.append(f"- {c}: ${m:,.0f} ({m / gastos * 100:.1f}%)")
    if gastos_prev > 0:
        dif = (gastos - gastos_prev) / gastos_prev * 100
        lineas.append(f"Comparación con {prev}: gastos {dif:+.1f}% (${gastos_prev:,.0f} el mes anterior).")
    else:
        lineas.append(f"Sin datos de {prev} para comparar.")
    return "\n".join(lineas)

In [ ]:
# Datos simulados del mes anterior
registrar_ingreso.invoke({"monto": 800000, "fuente": "Sueldo", "fecha": "2026-09-05"})
registrar_gasto.invoke({"monto": 300000, "categoria": "Servicios", "descripcion": "Arriendo", "fecha": "2026-09-05"})
registrar_gasto.invoke({"monto": 200000, "categoria": "Alimentación", "descripcion": "Supermercado", "fecha": "2026-09-10"})
registrar_gasto.invoke({"monto": 80000, "categoria": "Entretenimiento", "descripcion": "Salidas", "fecha": "2026-09-20"})

print(generar_alerta_categoria.invoke({}))
print("---")
print(generar_reporte_mensual.invoke({}))

ALERTA en Alimentación: $270,000 en 2026-10, 35% más que en 2026-09 ($200,000).
ALERTA en Entretenimiento: $180,000 en 2026-10, 125% más que en 2026-09 ($80,000).
---
REPORTE 2026-10
Ingresos: $800,000
Gastos: $751,200
Ahorro: $48,800 (6.1% del ingreso)
Gastos por categoría:
- Servicios: $300,000 (39.9%)
- Alimentación: $270,000 (35.9%)
- Entretenimiento: $180,000 (24.0%)
- Transporte: $1,200 (0.2%)
Comparación con 2026-09: gastos +29.5% ($580,000 el mes anterior).


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

herramientas = [registrar_gasto, registrar_ingreso, crear_meta_ahorro,
                consultar_gastos, buscar_guias_financieras,
                detectar_gastos_hormiga, calcular_plan_ahorro,
                generar_alerta_categoria, generar_reporte_mensual]

llm_agente = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite").bind_tools(herramientas)

def extraer_texto_limpio(contenido) -> str:
    if isinstance(contenido, str):
        return contenido
    return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in contenido)

def prompt_sistema() -> SystemMessage:
    return SystemMessage(content=(
        "Eres Sinanzas, un asistente de finanzas personales para usuarios en Chile. "
        "Responde siempre en español, de forma clara y breve. "
        "Usa las herramientas para registrar y consultar datos; nunca inventes cifras. "
        "Si falta un dato necesario (monto, categoría), pregúntalo antes de actuar. "
        f"La fecha de hoy es {hoy_chile()} (formato YYYY-MM-DD)."))

def nodo_agente(state: MessagesState):
    respuesta = llm_agente.invoke([prompt_sistema()] + state["messages"])
    return {"messages": [respuesta]}

grafo = StateGraph(MessagesState)
grafo.add_node("agente", nodo_agente)
grafo.add_node("herramientas", ToolNode(herramientas))
grafo.add_edge(START, "agente")
grafo.add_conditional_edges("agente", tools_condition,
                            {"tools": "herramientas", END: END})
grafo.add_edge("herramientas", "agente")

agente = grafo.compile(checkpointer=MemorySaver())

def chatear(texto: str, thread_id: str = "usuario_1") -> str:
    config = {"configurable": {"thread_id": thread_id}}
    res = agente.invoke({"messages": [HumanMessage(content=texto)]}, config=config)
    msgs = res["messages"]
    ult = max(i for i, m in enumerate(msgs) if isinstance(m, HumanMessage))
    for m in msgs[ult:]:
        for tc in getattr(m, "tool_calls", None) or []:
            print(f"[herramienta] {tc['name']}({tc['args']})")
    return extraer_texto_limpio(msgs[-1].content)

In [ ]:
print(chatear("Hoy gasté $4.500 en un café"))
print("---")
print(chatear("Anota otro gasto igual, pero de ayer"))
print("---")
print(chatear("¿Cómo voy este mes? Dame el reporte"))

[herramienta] registrar_gasto({'monto': 4500, 'descripcion': 'Café', 'categoria': 'Alimentación'})
Listo, he registrado tu gasto de $4.500 en Alimentación (Café).
---
[herramienta] registrar_gasto({'descripcion': 'Café', 'monto': 4500, 'fecha': '2026-10-24', 'categoria': 'Alimentación'})
Listo, he registrado el gasto de $4.500 en Alimentación (Café) con fecha de ayer (24 de octubre).
---
[herramienta] generar_reporte_mensual({'mes': '2026-10'})
Aquí tienes el resumen de tus finanzas para este mes (octubre de 2026):

* **Ingresos:** $800.000
* **Gastos totales:** $760.200
* **Ahorro:** $39.800 (5% de tus ingresos)

**Gastos por categoría:**
* **Servicios:** $300.000 (39,5%)
* **Alimentación:** $279.000 (36,7%)
* **Entretenimiento:** $180.000 (23,7%)
* **Transporte:** $1.200 (0,2%)

⚠️ Tus gastos aumentaron un **31,1%** en comparación con septiembre ($580.000).


In [ ]:
def cargar_datos_demo():
    """Borra los datos y carga un conjunto simulado coherente (septiembre y octubre)."""
    with sqlite3.connect(DB_PATH) as conn:
        for tabla in ("gastos", "ingresos", "metas"):
            conn.execute(f"DELETE FROM {tabla}")
        conn.execute("DELETE FROM sqlite_sequence")

    ingresos = [(800000, "Sueldo", "2026-09-05"), (800000, "Sueldo", "2026-10-05")]
    gastos = [
        # Septiembre
        (300000, "Servicios", "Arriendo", "2026-09-05"),
        (200000, "Alimentación", "Supermercado", "2026-09-10"),
        (80000, "Entretenimiento", "Salidas", "2026-09-20"),
        # Octubre
        (300000, "Servicios", "Arriendo", "2026-10-05"),
        (250000, "Alimentación", "Supermercado", "2026-10-10"),
        (180000, "Entretenimiento", "Salidas", "2026-10-15"),
        (1200, "Transporte", "Metro", "2026-10-12"),
    ] + [(2500, "Alimentación", "Café", f"2026-10-{d:02d}") for d in (6, 8, 12, 14, 16)]

    for monto, fuente, fecha in ingresos:
        registrar_ingreso.invoke({"monto": monto, "fuente": fuente, "fecha": fecha})
    for monto, cat, desc, fecha in gastos:
        registrar_gasto.invoke({"monto": monto, "categoria": cat, "descripcion": desc, "fecha": fecha})
    crear_meta_ahorro.invoke({"nombre": "Notebook nuevo", "monto_objetivo": 600000, "plazo_meses": 6})
    print("Datos de demo cargados.")

cargar_datos_demo()

Datos de demo cargados.


In [ ]:
# --- Memoria de largo plazo: SQLite (fuente de verdad) + FAISS (búsqueda semántica) ---
class MemoriaLargoPlazo:
    def __init__(self):
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute("""CREATE TABLE IF NOT EXISTS memorias (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                texto TEXT NOT NULL UNIQUE,
                fecha TEXT NOT NULL)""")
        self.embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
        self.indice = None
        textos = self.listar()
        if textos:
            self.indice = FAISS.from_texts(textos, self.embeddings)

    def listar(self):
        with sqlite3.connect(DB_PATH) as conn:
            return [f[0] for f in conn.execute("SELECT texto FROM memorias ORDER BY id")]

    def guardar(self, texto: str) -> bool:
        with sqlite3.connect(DB_PATH) as conn:
            try:
                conn.execute("INSERT INTO memorias (texto, fecha) VALUES (?, ?)", (texto, hoy_chile()))
            except sqlite3.IntegrityError:
                return False  # ya existía
        if self.indice is None:
            self.indice = FAISS.from_texts([texto], self.embeddings)
        else:
            self.indice.add_texts([texto])
        return True

    def buscar(self, consulta: str, k: int = 3):
        if self.indice is None:
            return []
        return [d.page_content for d in self.indice.similarity_search(consulta, k=k)]

memoria_lp = MemoriaLargoPlazo()

class MemoriaInput(BaseModel):
    texto: str = Field(description="Dato duradero sobre el usuario, en una frase breve "
                                   "(preferencia, objetivo o situación financiera estable)")

@tool("guardar_memoria", args_schema=MemoriaInput)
def guardar_memoria(texto: str) -> str:
    """Guarda en la memoria de largo plazo un dato duradero del usuario.
    Úsala cuando cuente una preferencia, un objetivo o algo que valga recordar
    en futuras conversaciones. No la uses para registrar gastos."""
    return "Recuerdo guardado." if memoria_lp.guardar(texto) else "Ya lo tenía anotado."

# --- Agente con recuperación automática de contexto ---
class EstadoSinanzas(MessagesState):
    contexto: str

herramientas_v2 = herramientas + [guardar_memoria]
llm_agente_v2 = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite").bind_tools(herramientas_v2)

def nodo_recuperar_contexto(state: EstadoSinanzas):
    consulta = extraer_texto_limpio(state["messages"][-1].content)
    recuerdos = memoria_lp.buscar(consulta, k=3)
    return {"contexto": "\n".join(f"- {r}" for r in recuerdos)}

def nodo_agente_v2(state: EstadoSinanzas):
    contexto = state.get("contexto") or "(sin recuerdos relevantes)"
    sistema = SystemMessage(content=(
        prompt_sistema().content +
        f"\n\nRecuerdos del usuario (memoria de largo plazo):\n{contexto}\n"
        "Si el usuario cuenta una preferencia u objetivo duradero, guárdalo con guardar_memoria. "
        "No guardes datos sensibles (salud, claves, números de tarjeta) ni montos de gastos."))
    return {"messages": [llm_agente_v2.invoke([sistema] + state["messages"])]}

grafo2 = StateGraph(EstadoSinanzas)
grafo2.add_node("recuperar_contexto", nodo_recuperar_contexto)
grafo2.add_node("agente", nodo_agente_v2)
grafo2.add_node("herramientas", ToolNode(herramientas_v2))
grafo2.add_edge(START, "recuperar_contexto")
grafo2.add_edge("recuperar_contexto", "agente")
grafo2.add_conditional_edges("agente", tools_condition, {"tools": "herramientas", END: END})
grafo2.add_edge("herramientas", "agente")

agente = grafo2.compile(checkpointer=MemorySaver())  # reemplaza al agente anterior

In [ ]:
# Conversación A: el usuario cuenta datos duraderos
print(chatear("Me pagan el día 5 de cada mes y quiero ahorrar para un viaje a Japón. "
              "Prefiero ahorros pequeños y graduales.", thread_id="conversacion_A"))
print("---")

# Conversación B: hilo nuevo, sin memoria de corto plazo
print(chatear("¿Cuándo me pagan y para qué estoy ahorrando?", thread_id="conversacion_B"))
print("---")
print("Recuerdos guardados:", memoria_lp.listar())

[herramienta] guardar_memoria({'texto': 'El usuario cobra su sueldo el día 5 de cada mes y su objetivo es ahorrar para un viaje a Japón prefiriendo ahorros pequeños y graduales.'})
¡Excelente objetivo! Viajar a Japón es una gran meta. 

Para poder calcular y armar un plan de ahorro realista para tu viaje (con montos pequeños y graduales), ¿podrías decirme **cuánto te gustaría juntar en total** y en **cuántos meses** te gustaría lograrlo (o para cuándo tienes pensado viajar)?
---
Según mis registros, cobras tu sueldo el **día 5 de cada mes** y tu objetivo es **ahorrar para un viaje a Japón** mediante ahorros pequeños y graduales.
---
Recuerdos guardados: ['El usuario cobra su sueldo el día 5 de cada mes y su objetivo es ahorrar para un viaje a Japón prefiriendo ahorros pequeños y graduales.']


In [ ]:
print("--- PRUEBA 1: Crear meta ---")
print(chatear("Quiero crear una meta de ahorro para un viaje a Japón", thread_id="prueba_plan"))

print("\n--- PRUEBA 2: Dar los datos de la meta ---")
print(chatear("Quiero ahorrar 1.000.000 en 6 meses", thread_id="prueba_plan"))

print("\n--- PRUEBA 3: Evaluar la meta (Planificación) ---")
print(chatear("¿Puedo cumplir esa meta con mis ingresos actuales?", thread_id="prueba_plan"))

--- PRUEBA 1: Crear meta ---
[herramienta] crear_meta_ahorro({'plazo_meses': 12, 'nombre': 'Viaje a Japón', 'monto_objetivo': 2000000})
¡Excelente iniciativa! Como tienes como objetivo ahorrar para un viaje a Japón, he registrado tu meta de ahorro:

* **Meta:** Viaje a Japón
* **Monto objetivo:** $2.000.000 CLP
* **Plazo:** 12 meses
* **Ahorro mensual sugerido:** $166.667 CLP

¿Te gustaría que calculemos si este monto mensual calza con tus ingresos y gastos actuales, o prefieres revisar algún plan de recortes?

--- PRUEBA 2: Dar los datos de la meta ---
[herramienta] crear_meta_ahorro({'monto_objetivo': 1000000, 'nombre': 'Viaje a Japón', 'plazo_meses': 6})
¡Perfecto! He actualizado tu meta:

* **Meta:** Viaje a Japón
* **Monto objetivo:** $1.000.000 CLP
* **Plazo:** 6 meses
* **Ahorro mensual requerido:** $166.667 CLP

Dado que tu objetivo es un ahorro pequeño y gradual, un poco más de $166 mil al mes es una excelente cifra para alcanzar tu millón en medio año. 

¿Quieres que analicem

In [ ]:
from typing import List, Literal
from langchain_core.messages import AIMessage

MAX_REPLANES = 1   # límite de replanificaciones (control de costo y latencia)
MAX_PASOS = 5

llm_base = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

class Clasificacion(BaseModel):
    ruta: Literal["simple", "plan"] = Field(description="'simple' o 'plan'")

class PlanPasos(BaseModel):
    pasos: List[str] = Field(description="Pasos ordenados por prioridad, de 2 a 5, "
                                         "cada uno una instrucción concreta")

class Decision(BaseModel):
    accion: Literal["continuar", "replanificar", "terminar"]
    motivo: str = Field(description="Justificación breve de la decisión")

clasificador = llm_base.with_structured_output(Clasificacion)
planificador_llm = llm_base.with_structured_output(PlanPasos)
evaluador_llm = llm_base.with_structured_output(Decision)

DESCRIPCION_HERRAMIENTAS = "\n".join(
    f"- {t.name}: {t.description.splitlines()[0]}" for t in herramientas_v2)

# ReAct sin memoria, usado para ejecutar un paso del plan
agente_paso = grafo2.compile()

class EstadoPlan(MessagesState):
    contexto: str
    ruta: str
    plan: List[str]
    indice: int
    resultados: List[str]
    replanes: int
    feedback: str
    decision: str

def ultima_peticion(state) -> str:
    return extraer_texto_limpio(state["messages"][-1].content)

def mostrar_herramientas(msgs):
    for m in msgs:
        for tc in getattr(m, "tool_calls", None) or []:
            print(f"   [herramienta] {tc['name']}({tc['args']})")

def nodo_router(state: EstadoPlan):
    c = clasificador.invoke(
        "Clasifica la petición de un usuario de una app de finanzas personales.\n"
        "- 'simple': una acción directa (registrar o consultar un dato, generar un reporte, "
        "responder una duda puntual, o contestar una pregunta anterior del asistente).\n"
        "- 'plan': requiere varias etapas o análisis combinado (cumplir una meta de ahorro, "
        "revisar la situación general, decidir recortes).\n"
        f"Petición: {ultima_peticion(state)}")
    print(f"[router] ruta = {c.ruta}")
    return {"ruta": c.ruta, "plan": [], "indice": 0, "resultados": [],
            "replanes": 0, "feedback": "", "decision": ""}

def nodo_planificador(state: EstadoPlan):
    solicitud = ultima_peticion(state)
    es_replan = bool(state.get("resultados"))
    prompt = (
        f"Eres el planificador de Sinanzas (finanzas personales). Hoy es {hoy_chile()}.\n"
        f"Herramientas disponibles:\n{DESCRIPCION_HERRAMIENTAS}\n\n"
        f"Recuerdos del usuario:\n{state.get('contexto') or '(ninguno)'}\n\n"
        f"Petición del usuario: {solicitud}\n\n"
        "Crea un plan de 2 a 5 pasos ordenados por prioridad: primero verificar datos y "
        "estado actual, luego analizar, al final recomendar. Cada paso debe ser una "
        "instrucción concreta que se resuelva con las herramientas.")
    if es_replan:
        prompt += ("\n\nYa se ejecutaron estos pasos:\n" + "\n".join(state["resultados"]) +
                   f"\n\nMotivo para replanificar: {state['feedback']}\n"
                   "Crea SOLO los pasos que faltan, adaptados a lo ya descubierto; "
                   "no repitas pasos hechos.")
    pasos = planificador_llm.invoke(prompt).pasos[:MAX_PASOS] or [solicitud]
    print("[replanificación]" if es_replan else "[plan]")
    for i, p in enumerate(pasos, 1):
        print(f"   {i}. {p}")
    return {"plan": pasos, "indice": 0,
            "replanes": state.get("replanes", 0) + (1 if es_replan else 0)}

def nodo_ejecutor(state: EstadoPlan):
    i = state["indice"]
    paso = state["plan"][i]
    print(f"[ejecutando paso {i + 1}/{len(state['plan'])}] {paso}")
    res = agente_paso.invoke({"messages": [HumanMessage(content=(
        "Ejecuta este paso de un plan usando las herramientas necesarias y reporta "
        f"el resultado con cifras exactas. Paso: {paso}"))]})
    mostrar_herramientas(res["messages"])
    texto = extraer_texto_limpio(res["messages"][-1].content)
    return {"resultados": state["resultados"] + [f"Paso: {paso}\nResultado: {texto}"],
            "indice": i + 1}

def nodo_evaluador(state: EstadoPlan):
    quedan = len(state["plan"]) - state["indice"]
    siguiente = state["plan"][state["indice"]] if quedan else "(ninguno)"
    d = evaluador_llm.invoke(
        f"Petición original: {ultima_peticion(state)}\n"
        f"Último resultado:\n{state['resultados'][-1]}\n"
        f"Pasos pendientes: {quedan}. Próximo paso: {siguiente}\n\n"
        "Decide: 'continuar' si el resultado es normal y el plan sigue siendo válido; "
        "'replanificar' si el resultado revela algo que cambia el plan (la meta no es "
        "alcanzable, faltan datos, hay un gasto fuera de lo normal); 'terminar' si ya hay "
        "información suficiente para responder.")
    accion = d.accion
    if accion == "replanificar" and state["replanes"] >= MAX_REPLANES:
        accion = "continuar" if quedan else "terminar"
    if accion == "continuar" and quedan == 0:
        accion = "terminar"
    print(f"[decisión] {accion}: {d.motivo}")
    return {"decision": accion, "feedback": d.motivo}

def nodo_sintetizador(state: EstadoPlan):
    resp = llm_base.invoke(
        f"Petición del usuario: {ultima_peticion(state)}\n\n"
        "Resultados de los pasos ejecutados:\n" + "\n\n".join(state["resultados"]) +
        "\n\nRedacta la respuesta final en español, breve y clara, con cifras exactas y "
        "recomendaciones concretas. No inventes datos que no estén en los resultados.")
    return {"messages": [AIMessage(content=extraer_texto_limpio(resp.content))]}

grafo3 = StateGraph(EstadoPlan)
grafo3.add_node("recuperar_contexto", nodo_recuperar_contexto)
grafo3.add_node("router", nodo_router)
grafo3.add_node("agente", nodo_agente_v2)
grafo3.add_node("herramientas", ToolNode(herramientas_v2))
grafo3.add_node("planificador", nodo_planificador)
grafo3.add_node("ejecutor", nodo_ejecutor)
grafo3.add_node("evaluador", nodo_evaluador)
grafo3.add_node("sintetizador", nodo_sintetizador)

grafo3.add_edge(START, "recuperar_contexto")
grafo3.add_edge("recuperar_contexto", "router")
grafo3.add_conditional_edges("router", lambda s: s["ruta"],
                             {"simple": "agente", "plan": "planificador"})
grafo3.add_conditional_edges("agente", tools_condition, {"tools": "herramientas", END: END})
grafo3.add_edge("herramientas", "agente")
grafo3.add_edge("planificador", "ejecutor")
grafo3.add_edge("ejecutor", "evaluador")
grafo3.add_conditional_edges("evaluador", lambda s: s["decision"],
                             {"continuar": "ejecutor", "replanificar": "planificador",
                              "terminar": "sintetizador"})
grafo3.add_edge("sintetizador", END)

agente = grafo3.compile(checkpointer=MemorySaver())  # reemplaza al agente anterior

In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter

# 0,2 solicitudes por segundo = 12 por minuto, bajo el límite gratuito de 15
limitador = InMemoryRateLimiter(requests_per_second=0.2,
                                check_every_n_seconds=0.2,
                                max_bucket_size=1)

llm_agente_v2 = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite",
                                       rate_limiter=limitador).bind_tools(herramientas_v2)
llm_base = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", rate_limiter=limitador)
clasificador = llm_base.with_structured_output(Clasificacion)
planificador_llm = llm_base.with_structured_output(PlanPasos)
evaluador_llm = llm_base.with_structured_output(Decision)

In [ ]:
# A) Reinicio de demo que también limpia la memoria de largo plazo
def reiniciar_demo():
    cargar_datos_demo()
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("DELETE FROM memorias")
    memoria_lp.indice = None
    print("Memoria de largo plazo reiniciada.")

# B) El ejecutor de pasos solo puede consultar y calcular (mínimo privilegio)
herramientas_analisis = [consultar_gastos, buscar_guias_financieras,
                         detectar_gastos_hormiga, calcular_plan_ahorro,
                         generar_alerta_categoria, generar_reporte_mensual]

llm_paso = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite",
                                  rate_limiter=limitador).bind_tools(herramientas_analisis)

def nodo_paso(state: MessagesState):
    sistema = SystemMessage(content=(
        prompt_sistema().content +
        "\nEstás ejecutando UN paso de un plan de análisis. Solo consulta y calcula; "
        "no registres ni modifiques datos. Si falta un dato para ejecutar el paso "
        "(por ejemplo, una meta que no existe), NO lo inventes: responde exactamente "
        "'FALTA DATO: ' seguido de qué falta."))
    return {"messages": [llm_paso.invoke([sistema] + state["messages"])]}

g_paso = StateGraph(MessagesState)
g_paso.add_node("agente", nodo_paso)
g_paso.add_node("herramientas", ToolNode(herramientas_analisis))
g_paso.add_edge(START, "agente")
g_paso.add_conditional_edges("agente", tools_condition, {"tools": "herramientas", END: END})
g_paso.add_edge("herramientas", "agente")
agente_paso = g_paso.compile()

In [ ]:
reiniciar_demo()   # parte de un estado limpio y conocido

print("=== PRUEBA 1: petición compleja, la meta NO es alcanzable ===")
print(chatear("Quiero cumplir mi meta del notebook. Revisa cómo voy y dime qué hacer.",
              thread_id="plan_1"))

print("\n=== PRUEBA 2: cambia la condición (llega un bono) ===")
registrar_ingreso.invoke({"monto": 300000, "fuente": "Bono", "fecha": hoy_chile()})
print(chatear("Quiero cumplir mi meta del notebook. Revisa cómo voy y dime qué hacer.",
              thread_id="plan_2"))

print("\n=== PRUEBA 3: petición simple ===")
print(chatear("Gasté $3.000 en transporte hoy", thread_id="simple_1"))

Datos de demo cargados.
Memoria de largo plazo reiniciada.
=== PRUEBA 1: petición compleja, la meta NO es alcanzable ===
[router] ruta = plan
[plan]
   1. consultar_gastos
   2. generar_reporte_mensual
   3. calcular_plan_ahorro
[ejecutando paso 1/3] consultar_gastos
   [herramienta] consultar_gastos({'mes': '2026-10'})
[decisión] continuar: Se han consultado los gastos correctamente y se debe proceder a generar el reporte mensual para evaluar la meta del notebook.
[ejecutando paso 2/3] generar_reporte_mensual
   [herramienta] generar_reporte_mensual({'mes': '2026-10'})
[decisión] continuar: El reporte muestra ingresos y gastos actuales, pero es necesario calcular el plan de ahorro para determinar si la meta del notebook sigue siendo alcanzable.
[ejecutando paso 3/3] calcular_plan_ahorro
   [herramienta] calcular_plan_ahorro({})
[decisión] replanificar: El disponible actual no cubre la cuota mensual necesaria para la meta y se requiere ajustar el plan mediante recortes o ampliación de 